<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session09-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 9: in-class discussion problem (1 of 3)

This is the block 1 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then run the code below and look at what it does. There is one mistake in it, but finding that is
only the way in: the point is to work out why the result comes out as it does. Discuss that with
your neighbour, then answer the quiz again.

## Which gradients reach the hidden unit?

A tiny network: two inputs, one **ReLU** hidden unit, one sigmoid output, and binary cross-entropy
loss. The book page did the same by hand with a tanh hidden unit. Here the weights are chosen so that
the hidden unit's input is negative:

$z_1 = w_1 x_1 + w_2 x_2 + b_1 = 0.2 \cdot 0.5 + 0.4 \cdot (-0.3) - 0.5 = -0.52$, and $a_1 = \text{ReLU}(z_1)$.

After `loss.backward()`, what is `w1.grad`?

**Fix and run.** The first cell does the backward pass by hand, the second lets PyTorch's autograd do
it. Run both as they are: do the hand-computed gradients agree with autograd? The line marked `FIX`
uses the wrong local derivative for ReLU. Fix it, and run both cells again.

In [1]:
import math

x1, x2 = 0.5, -0.3
w1, w2, b1 = 0.2, 0.4, -0.5      # the hidden unit
w3, b2 = 0.9, -0.1               # the output unit
y_true = 1.0

# Forward pass
z1 = w1 * x1 + w2 * x2 + b1
a1 = max(0.0, z1)                # ReLU
z2 = w3 * a1 + b2
y_hat = 1 / (1 + math.exp(-z2))
loss = -math.log(y_hat)          # binary cross-entropy for y_true = 1
print(f"z1 = {z1:.2f}  a1 = {a1:.2f}  y_hat = {y_hat:.3f}  loss = {loss:.3f}")

# Backward pass by hand: the chain rule, from the loss back to each weight
dL_dz2 = y_hat - y_true          # sigmoid with cross-entropy
dL_dw3, dL_db2 = dL_dz2 * a1, dL_dz2
dL_da1 = dL_dz2 * w3
relu_deriv = 1.0                 # FIX: ReLU's derivative is not always 1
dL_dz1 = dL_da1 * relu_deriv
hand = {"w1": dL_dz1 * x1, "w2": dL_dz1 * x2, "b1": dL_dz1, "w3": dL_dw3, "b2": dL_db2}
for name, g in hand.items():
    print(f"by hand:  dL/d{name} = {g:+.4f}")

z1 = -0.52  a1 = 0.00  y_hat = 0.475  loss = 0.744
by hand:  dL/dw1 = -0.2362
by hand:  dL/dw2 = +0.1417
by hand:  dL/db1 = -0.4725
by hand:  dL/dw3 = -0.0000
by hand:  dL/db2 = -0.5250


In [2]:
import torch

p = {name: torch.tensor(v, dtype=torch.float64, requires_grad=True)
     for name, v in [("w1", w1), ("w2", w2), ("b1", b1), ("w3", w3), ("b2", b2)]}
z1_t = p["w1"] * x1 + p["w2"] * x2 + p["b1"]
y_hat_t = torch.sigmoid(p["w3"] * torch.relu(z1_t) + p["b2"])
loss_t = -torch.log(y_hat_t)
loss_t.backward()

print(f"{'':6}{'by hand':>10}{'autograd':>10}")
for name in hand:
    auto = p[name].grad.item()
    flag = "" if abs(hand[name] - auto) < 1e-8 else "   <- disagree"
    print(f"{name:6}{hand[name]:>+10.4f}{auto:>+10.4f}{flag}")

         by hand  autograd
w1       -0.2362   +0.0000   <- disagree
w2       +0.1417   -0.0000   <- disagree
b1       -0.4725   +0.0000   <- disagree
w3       -0.0000   -0.0000
b2       -0.5250   -0.5250


**If you finish early.**

1. Change `b1` to `+0.5`, so that $z_1$ is positive. Which gradients change, and do hand and autograd
   still agree?
2. If $z_1$ is negative for *every* training example, what happens to this hidden unit during
   training? Would a tanh unit have the same problem?

<details>
<summary><b>Discussion point</b> &mdash; open this once you have run the code and talked it through</summary>

As it was, the hand calculation treated ReLU as if its derivative were
always 1, and got $\partial L/\partial w_1 = -0.24$: a plausible number, and wrong. Autograd said 0,
and the two disagreed for $w_1$, $w_2$ and $b_1$. The fix is

```python
relu_deriv = 1.0 if z1 > 0 else 0.0
```

Then all five gradients agree. ReLU's derivative is 0 wherever its input is negative, so the error
signal stops at the hidden unit: $w_1$, $w_2$ and $b_1$ get gradient **exactly 0**, however large the
loss (0.744). Only the output bias $b_2$ learns on this example ($-0.525$); $w_3$ gets 0 too, because
the hidden unit's output $a_1$ is 0.

A tanh unit never has an exactly-zero derivative; its gradient only gets small for large $|z|$. A ReLU
unit whose input is negative for every example never learns again: a "dead" ReLU. Comparing a hand
calculation with autograd is how you catch mistakes in either one.

</details>